# DS 6021 — Hierarchical Clustering
## Interactive lecture companion · Intro to Predictive Modeling

**Learning goals:** Understand what distances and linkage rules do, follow a dendrogram, compare cuts using silhouette scores, connect Ward's method to K-means, and apply Gower dissimilarity to mixed data.

We use **synthetic viewer preferences** for *Breaking Bad* and *Love Is Blind* to connect this lesson to our previous K-means discussion. All ratings are fictional and are used only for instruction.

**Main idea:** Start with one viewer per cluster → merge closest groups → build dendrogram → decide where to cut. Unlike K-means, **we do not choose K before building the tree**.

### 1. Begin with four viewers: calculate the first distances ourselves

We have four people rating the same two shows from 1 to 5. Can you guess who merges first?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import silhouette_score, silhouette_samples

viewers4 = pd.DataFrame({
    "Breaking Bad": [1, 1, 4, 4],
    "Love Is Blind": [1, 2, 4, 5],
}, index=["A", "B", "C", "D"])

viewers4

In [ ]:
ax = viewers4.plot.scatter(x="Breaking Bad", y="Love Is Blind", s=110, figsize=(6, 5))
for name, row in viewers4.iterrows():
    ax.annotate(name, (row["Breaking Bad"] + 0.07, row["Love Is Blind"] + 0.07))
ax.set(xlim=(0.5, 5.5), ylim=(0.5, 5.5), title="Four viewers — which ones are closest?")
ax.grid(alpha=0.2)
plt.show()

In [ ]:
distances4 = pd.DataFrame(
    squareform(pdist(viewers4, metric="euclidean")),
    index=viewers4.index,
    columns=viewers4.index
)
distances4.round(2)

**Think:** Distance between A and B is 1; between C and D is 1. These are the first two merges (their order is a tie). But after the merges, **what is the distance between the groups {A,B} and {C,D}?** We now need a linkage rule.

### 2. Linkage means distance *between groups*

Four cross-cluster distances are involved. Compare **single** (smallest), **complete** (largest), and **average** (mean).

In [ ]:
cross_distances = distances4.loc[["A", "B"], ["C", "D"]]
print("Distances connecting the two groups:")
print(cross_distances.round(2))
print("Single linkage:", round(cross_distances.values.min(), 2))
print("Complete linkage:", round(cross_distances.values.max(), 2))
print("Average linkage:", round(cross_distances.values.mean(), 2))

In [ ]:
# Use one method at a time, just as in the dashboard.
method = "single"   # change to "complete", "average", or "ward"
Z4 = linkage(viewers4, method=method, metric="euclidean")

plt.figure(figsize=(8, 4))
dendrogram(Z4, labels=viewers4.index.tolist())
plt.ylabel("Linkage height")
plt.title(f"Dendrogram — {method} linkage")
plt.show()

# The linkage matrix records each merge and the height of that merge.
pd.DataFrame(Z4, columns=["Group 1", "Group 2", "Height", "New size"]).round(2)

**Read the dendrogram:** Each leaf is one viewer. A horizontal branch means two groups merged. Its **height is the chosen linkage quantity** (Ward is an exception: SciPy uses the square root of twice the SSE increase).

**Question:** If the data do not change, why does the height of the final merge change when you switch from single to complete linkage?

### 3. Cut the tree: choose K *after* fitting

In [ ]:
Z4 = linkage(viewers4, method="average")
clusters4 = fcluster(Z4, t=2, criterion="maxclust")

result4 = viewers4.copy()
result4["cluster"] = clusters4
result4

Changing `t=2` to a different number of groups changes the **cut**, not the hierarchy. With just four observations, different choices may be less informative. Next, we use a slightly larger example.

### 4. A larger set of fictional viewers

This is a fictional viewer dataset to learn the algorithm before we apply it to the live weather dashboard. Two numerical variables are on the same 1–5 rating scale; standardization can still be considered, but it is not automatically required.

In [ ]:
rating_pairs = [
    (1.0,4.6),(1.4,4.8),(1.6,4.3),(1.2,3.9),
    (4.4,1.0),(4.8,1.3),(4.3,1.6),(3.9,1.2),
    (3.4,4.3),(3.7,3.9),(4.1,4.7),(3.2,4.7),
    (2.2,3.2),(2.8,2.7)
]
habits = ["Binge","Binge","Weekly","Binge", "Weekly","Weekly","Binge","Weekly",
          "Binge","Weekly","Binge","Weekly", "Binge","Weekly"]

viewers = pd.DataFrame(rating_pairs, columns=["Breaking Bad", "Love Is Blind"])
viewers.index = [f"V{i+1:02d}" for i in range(len(viewers))]
viewers["Viewing habit"] = habits
viewers.head()

In [ ]:
X = viewers[["Breaking Bad", "Love Is Blind"]]
Z = linkage(X, method="average", metric="euclidean")

plt.figure(figsize=(11, 4))
dendrogram(Z, labels=viewers.index.tolist(), color_threshold=0)
plt.ylabel("Average-linkage distance")
plt.title("Streaming-viewer hierarchy")
plt.xticks(rotation=45)
plt.show()

In [ ]:
K = 3
viewers["cluster"] = fcluster(Z, t=K, criterion="maxclust")

fig, ax = plt.subplots(figsize=(7, 5))
for group, part in viewers.groupby("cluster"):
    ax.scatter(part["Breaking Bad"], part["Love Is Blind"], s=80, label=f"Cluster {group}")
    for name, row in part.iterrows():
        ax.annotate(name, (row["Breaking Bad"]+0.04, row["Love Is Blind"]+0.05), fontsize=8)
ax.set(xlabel="Breaking Bad rating", ylabel="Love Is Blind rating",
       title=f"Average linkage, K = {K}", xlim=(0.5,5.5), ylim=(0.5,5.5))
ax.legend()
plt.show()

**Interpret:** Describe the preferences of each cluster. What happens to the group memberships when you change `K=3` to `K=4`? Try `method="single"` and `method="complete"` and ask which groups seem coherent.

### 5. Which cut has the best average silhouette?

Recall our K-means lesson: for each viewer, silhouette compares the mean distance to their own group, $a(i)$, with the mean distance to the **nearest other group**, $b(i)$:

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.$$

In [ ]:
results = []
for k in range(2, 10):
    labels = fcluster(Z, t=k, criterion="maxclust")
    # Avoid invalid silhouette calculations if a tie gives fewer than k groups.
    if 1 < len(np.unique(labels)) < len(X):
        avg = silhouette_score(X, labels, metric="euclidean")
        results.append((k, avg))

silhouette_table = pd.DataFrame(results, columns=["K", "Average silhouette"])
silhouette_table.round(3)

In [ ]:
ax = silhouette_table.plot(x="K", y="Average silhouette", marker="o", legend=False, figsize=(7, 4))
ax.set(title="Average silhouette vs. K", ylabel="Average silhouette")
ax.grid(alpha=0.2)
plt.show()

best_k = int(silhouette_table.loc[silhouette_table["Average silhouette"].idxmax(), "K"])
print("Highest average silhouette at K =", best_k)

In [ ]:
K = 3
labels = fcluster(Z, t=K, criterion="maxclust")
individual_scores = silhouette_samples(X, labels, metric="euclidean")
profile = pd.DataFrame({"Viewer": X.index, "Cluster": labels, "Silhouette": individual_scores})
profile = profile.sort_values(["Cluster", "Silhouette"])

fig, ax = plt.subplots(figsize=(9, 5))
for group, subset in profile.groupby("Cluster"):
    ax.barh(subset["Viewer"], subset["Silhouette"], label=f"Cluster {group}")
ax.axvline(0, color="black", lw=1)
ax.set(xlabel="Individual silhouette score", ylabel="Viewer", title=f"Silhouette profiles for K = {K}")
ax.legend()
plt.show()

**Interpret:** An average is not the full story. Which viewers are on cluster boundaries or have negative silhouettes? If `K=4` has the highest average, must we choose 4? Explain how subject-matter interpretation and stability could matter.

### 6. Ward: hierarchical clustering with an SSE objective

For groups $A$ and $B$ with sizes $n_A,n_B$ and mean vectors $\bar{x}_A,\bar{x}_B$:

$$\Delta\mathrm{SSE}=\frac{n_A n_B}{n_A+n_B}\|\bar x_A-\bar x_B\|^2.$$

This is why Ward resembles K-means. K-means reassigns points for a chosen K; Ward greedily merges groups to minimize the *increase* in SSE.

In [ ]:
A = viewers4.loc[["A", "B"]].to_numpy()
B = viewers4.loc[["C", "D"]].to_numpy()

increase_in_sse = (len(A)*len(B)/(len(A)+len(B))) * np.sum((A.mean(axis=0)-B.mean(axis=0))**2)
print("Ward SSE increase when joining these two groups:", increase_in_sse)
print("Corresponding SciPy Ward dendrogram height:", np.sqrt(2*increase_in_sse))

Z_ward = linkage(X, method="ward")
plt.figure(figsize=(10, 4))
dendrogram(Z_ward, labels=viewers.index.tolist())
plt.title("Ward hierarchical clustering")
plt.ylabel("Ward height = sqrt(2 × SSE increase)")
plt.show()

### 7. Gower: numeric + categorical features

The two axes on the scatterplot cannot show the categorical feature. Gower averages (1) normalized numeric differences and (2) 0/1 categorical mismatches. For this small complete dataset, we can compute it in a transparent loop rather than hiding the logic in a library.

**Important:** Do **not** use Ward on the resulting Gower dissimilarity matrix; use single, complete, or average linkage.

In [ ]:
n = len(viewers)
D_gower = np.zeros((n, n))

x_range = viewers["Breaking Bad"].max() - viewers["Breaking Bad"].min()
y_range = viewers["Love Is Blind"].max() - viewers["Love Is Blind"].min()

for i in range(n):
    for j in range(n):
        d1 = abs(viewers.iloc[i]["Breaking Bad"] - viewers.iloc[j]["Breaking Bad"]) / x_range
        d2 = abs(viewers.iloc[i]["Love Is Blind"] - viewers.iloc[j]["Love Is Blind"]) / y_range
        d3 = 0 if viewers.iloc[i]["Viewing habit"] == viewers.iloc[j]["Viewing habit"] else 1
        D_gower[i, j] = (d1 + d2 + d3) / 3

pd.DataFrame(D_gower, index=viewers.index, columns=viewers.index).round(2).iloc[:6, :6]

In [ ]:
Z_gower = linkage(squareform(D_gower), method="average")

plt.figure(figsize=(11, 4))
dendrogram(Z_gower, labels=viewers.index.tolist())
plt.ylabel("Average Gower dissimilarity")
plt.title("Gower + average linkage")
plt.show()

labels_gower = fcluster(Z_gower, t=3, criterion="maxclust")
print("Average silhouette using Gower:",
      round(silhouette_score(D_gower, labels_gower, metric="precomputed"), 3))

**Reflection questions**

1. Compare numeric Euclidean + average linkage with Gower + average linkage. Why might a viewer switch groups?
2. Would it be fair to give a categorical variable the same weight as two numerical features? What judgment are we making?
3. Why is it not meaningful to calculate a centroid of a nominal category such as *Binge* / *Weekly*?
4. What rule would you need if a brand-new viewer arrives and wants a cluster assignment? (A dendrogram by itself does not prescribe one.)

**Suggested reading:** *Introduction to Statistical Learning with Applications in Python* (chapter 12, hierarchical clustering); optional: *Elements of Statistical Learning* on hierarchical clustering.

---

# 8. Apply hierarchical clustering to the Charlottesville Weather Dashboard

These cells reproduce the **Hierarchical Clustering tab** of our [DS 6021 weather dashboard](https://afriyiep1.github.io/ds6021-predictive-modeling/kmeans/#hier). The notebook uses the **same saved hourly weather snapshot** and the same preprocessing choices as the website. The dashboard can also refresh from Open-Meteo, so newly refreshed data may give different answers.

**Dashboard controls we will reproduce:**

| Dashboard control | Notebook setting |
|---|---|
| Sampled weather hours | `sample_n = 24, 48, 72, or 96` |
| Features | `dimensions = 2` (cloud, humidity), or `3` (+ precipitation) |
| Distance | `distance = "euclidean"` or `"gower"` |
| Linkage | `linkage_method = "single", "complete", "average", "ward"` |
| Variable scale | `standardize = True` for z-scores, or `False` for raw values (Euclidean only) |
| Cut | `K` between 2 and 10 (within valid sample range) |

**Important:** These are unsupervised clusters. We do **not** use the WMO target categories from the KNN/logit tabs to construct them. **Weather data are model estimates**, and consecutive hours are temporally dependent.

### 8.1 Load the same saved weather data

Open-Meteo hourly model estimates for Charlottesville, Virginia. For a reproducible class demonstration, use the version saved in the GitHub repository. This CSV includes more variables than we need; our clustering predictors are cloud cover, relative humidity, and optionally precipitation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, cut_tree
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score, silhouette_samples

url = "https://raw.githubusercontent.com/afriyiep1/ds6021-predictive-modeling/main/docs/kmeans/weather-snapshot.csv"
weather = pd.read_csv(url)
weather["local_time"] = pd.to_datetime(weather["time"], unit="s", utc=True).dt.tz_convert("America/New_York")

print("Saved weather hours:", len(weather))
print("Date range:", weather["local_time"].min(), "to", weather["local_time"].max())
weather[["local_time", "cloud_cover", "relative_humidity_2m", "precipitation"]].head()

**Optional: refresh from Open-Meteo instead of using the saved snapshot.** The dashboard's Refresh button requests 60 days of hourly model estimates for Charlottesville. Run the following cell **only if you want refreshed data** (then rerun the sections below). The saved snapshot is recommended for reproducible class discussions.

In [ ]:
# OPTIONAL: Uncomment this cell to get fresh hourly data, matching the dashboard API.
# import requests
#
# api_url = "https://api.open-meteo.com/v1/forecast"
# params = {
#     "latitude": 38.0293,
#     "longitude": -78.4767,
#     "hourly": "temperature_2m,relative_humidity_2m,precipitation,cloud_cover,weather_code",
#     "timezone": "UTC",
#     "timeformat": "unixtime",
#     "past_days": 60,
#     "forecast_days": 0
# }
# response = requests.get(api_url, params=params, timeout=30)
# response.raise_for_status()
# weather = pd.DataFrame(response.json()["hourly"])
# weather["local_time"] = pd.to_datetime(weather["time"], unit="s", utc=True).dt.tz_convert("America/New_York")
# print("Refreshed", len(weather), "weather hours")

### 8.2 Match the dashboard's options and sample

The browser does not cluster every hour, because a full distance matrix is costly for teaching interactions. It selects **evenly spaced complete observations** across the active weather series. The code below uses the same deterministic selection rule. It does not randomly sample observations.

In [ ]:
# Change these settings to match the controls in the Hierarchical tab.
sample_n = 48                    # 24, 48, 72, or 96
dimensions = 2                  # 2D, or 3D
standardize = True             # z-scores vs raw Euclidean values
distance = "euclidean"         # "euclidean" or "gower"
linkage_method = "average"     # "single", "complete", "average", "ward"
K = 3                          # clusters obtained by cutting the full tree

features = ["cloud_cover", "relative_humidity_2m"]
if dimensions == 3:
    features.append("precipitation")

valid = weather.dropna(subset=["time", "cloud_cover", "relative_humidity_2m", "precipitation"]).copy()
n = min(sample_n, len(valid))
selected_indices = np.floor((np.arange(n) + 0.5) * len(valid) / n).astype(int)
sampled = valid.iloc[selected_indices].reset_index(drop=True)

print("Sampled weather hours:", len(sampled))
print("Features:", features)
sampled[["local_time"] + features].head()

### 8.3 Distances: Raw Euclidean, Z-score Euclidean, or mixed-type Gower

- **Raw Euclidean:** numerically larger variables may dominate distances.
- **Standardized Euclidean:** each feature is transformed to a z-score using the **selected sample** (population SD, `ddof=0`). Plot axes use these same z-score coordinates.
- **Gower:** each numerical feature is min-max normalized; a categorical **daypart** (four 6-hour intervals in Charlottesville local time) contributes 0 for a match and 1 for a mismatch. All contributions are averaged equally. The Gower scatterplot shows raw numeric weather units, as the categorical daypart cannot be drawn on the 2D axes.

Ward's SSE formulation requires Euclidean features, so we do not combine Ward with Gower.

In [ ]:
raw = sampled[features].astype(float)
means = raw.mean()
std = raw.std(ddof=0).replace(0, 1)
z_scores = (raw - means) / std

if distance == "euclidean":
    X_weather = z_scores if standardize else raw
    D_weather = squareform(pdist(X_weather, metric="euclidean"))
    print("Clustering with", "z-scores" if standardize else "raw units")
else:
    if linkage_method == "ward":
        raise ValueError("Ward requires Euclidean feature coordinates, not Gower dissimilarities.")
    ranges = (raw.max() - raw.min()).replace(0, 1)
    scaled = (raw - raw.min()) / ranges
    daypart = sampled["local_time"].dt.hour // 6
    n = len(sampled)
    D_weather = np.zeros((n, n))
    for i in range(n):
        for j in range(i):
            differences = np.abs(scaled.iloc[i].to_numpy() - scaled.iloc[j].to_numpy()).sum()
            daypart_mismatch = int(daypart.iloc[i] != daypart.iloc[j])
            D_weather[i, j] = (differences + daypart_mismatch) / (len(features) + 1)
            D_weather[j, i] = D_weather[i, j]
    # The scatterplot shows original values, while clustering uses the Gower matrix.
    X_weather = raw
    print("Gower distance: numeric features + categorical daypart")

print("Distance between the first two sampled hours:", round(D_weather[0, 1], 3))

### 8.4 Build the tree: Which two groups merge first?

SciPy uses a linkage matrix. Each row records (1) the merged groups, (2) their merge **height**, and (3) the number of observations now in the merged group. The heights depend on the distance and linkage rule. Under **Ward**, SciPy's height is √(2 × the increase in SSE).

In [ ]:
if distance == "euclidean":
    Z_weather = linkage(X_weather.to_numpy(), method=linkage_method, metric="euclidean")
else:
    Z_weather = linkage(squareform(D_weather), method=linkage_method)

merges = pd.DataFrame(Z_weather, columns=["Left group", "Right group", "Merge height", "Group size"])
merges.index = np.arange(1, len(merges) + 1)
print("First five merges:")
print(merges.head().round(3))

plt.figure(figsize=(12, 4))
dendrogram(Z_weather, no_labels=True, color_threshold=0, above_threshold_color="#232D4B")
plt.title("Weather dendrogram · " + linkage_method + " linkage · " + distance)
plt.xlabel("Sampled weather hours (leaf order)")
plt.ylabel("Ward height" if linkage_method == "ward" else "Linkage distance")
plt.show()

### 8.5 Cut at K and inspect the weather groups

Cutting **does not rebuild** the tree. We use `cut_tree` to request exactly K groups even if two merges occur at the same height. The cluster numbers are arbitrary, not WMO sky-condition labels.

In [ ]:
K = 3
cluster = cut_tree(Z_weather, n_clusters=[K]).ravel() + 1
sampled["cluster"] = cluster

# The scatterplot uses the SAME coordinates that created Euclidean distances.
# Gower is a special case: the axes show raw numeric variables, but the clustering
# also uses the categorical daypart, which cannot be plotted on two numeric axes.
fig, ax = plt.subplots(figsize=(7, 5))
for group in sorted(sampled["cluster"].unique()):
    idx = sampled["cluster"] == group
    ax.scatter(X_weather.loc[idx, features[0]], X_weather.loc[idx, features[1]],
               label="Cluster " + str(group), s=55, alpha=0.8)
ax.set_xlabel(features[0] + (" (z-score)" if distance == "euclidean" and standardize else " (original units)"))
ax.set_ylabel(features[1] + (" (z-score)" if distance == "euclidean" and standardize else " (original units)"))
ax.set_title("Weather groups at K = " + str(K))
ax.legend()
plt.show()

print("Weather profiles in original units:")
print(sampled.groupby("cluster")[features].agg(["count", "mean"]).round(2))

### 8.6 Select K using the same silhouette logic as the dashboard

Silhouette scores use the **actual pairwise distances used for clustering**. For example, Gower uses a precomputed Gower distance matrix rather than Euclidean distances on the displayed scatter axes. Averages can conceal individual poor fits; inspect the profile at selected K.

In [ ]:
summary = []
for k in range(2, min(10, len(sampled) - 1) + 1):
    groups_k = cut_tree(Z_weather, n_clusters=[k]).ravel()
    mean_silhouette = silhouette_score(D_weather, groups_k, metric="precomputed")
    summary.append({"K": k, "Average silhouette": mean_silhouette})

silhouette_table_weather = pd.DataFrame(summary)
print(silhouette_table_weather.round(3).to_string(index=False))

ax = silhouette_table_weather.plot(x="K", y="Average silhouette", marker="o", figsize=(7, 4), legend=False)
ax.axvline(K, color="gray", linestyle="--", label="Selected K")
ax.set(title="Weather: silhouette vs. K", ylabel="Average silhouette")
ax.set_xticks(silhouette_table_weather["K"])
ax.legend()
plt.show()

In [ ]:
labels = cut_tree(Z_weather, n_clusters=[K]).ravel()
s_i = silhouette_samples(D_weather, labels, metric="precomputed")
profiles = pd.DataFrame({"Cluster": labels + 1, "Silhouette": s_i})
profiles = profiles.sort_values(["Cluster", "Silhouette"])

fig, ax = plt.subplots(figsize=(8, 5))
for group in sorted(profiles["Cluster"].unique()):
    scores = profiles.loc[profiles["Cluster"] == group, "Silhouette"].to_numpy()
    ax.barh(np.arange(len(scores)) + (group - 1) * len(sampled), scores,
            height=0.8, label="Cluster " + str(group))
ax.axvline(0, color="black", linewidth=1)
ax.set(xlim=(-1, 1), xlabel="Individual silhouette", ylabel="Observations grouped by cluster")
ax.legend()
plt.show()

print("Mean silhouette at selected K:", round(s_i.mean(), 3))
print("Negative silhouettes:", int((s_i < 0).sum()))

### 8.7 Classroom challenges

1. Set `standardize=False` and rebuild. Compare the scale and range of the axes, the merge heights, and the clusters.
2. Keep `sample_n=48`, `dimensions=2` and `distance="euclidean"`. Compare single, complete, average, and Ward linkages. Does the best K change?
3. Set `dimensions=3` to include precipitation. Which weather groups change? Check the raw profiles to interpret the new groups.
4. Set `distance="gower"` and `linkage_method="average"` to add the categorical daypart. Why do the plotted raw axes not fully explain cluster membership?
5. On the dashboard, advance a few merges. After changing K, did the **merge tree** change or only the **horizontal cut**?
6. Are groups discovered by clustering necessarily equivalent to the WMO sky-condition targets used for KNN and logistic regression? Explain.

**Reproducibility note:** The dashboard uses a JavaScript implementation with a deterministic tie-break rule. SciPy may choose a different but equally valid merge order when distances tie; cluster numbering also may differ.